In [1]:
from datetime import datetime, timedelta
from os import getenv

import pandas as pd

from utils import get_candles, get_stocks


В этом модуле мы сформируем .scv файлы, чтобы постоянно с нуля не формировать датафрейм. Для получения данных используем SDK Т-Банка

In [2]:
# выгружаю токен для т-инвестиций
TOKEN = getenv("t_bank_token")
# P.S: если захотите воспользоваться этим модулем, вам придётся создать свой токен на сайте т-банка
# и записать его в эту переменную в таком формате: TOKEN = "ваш_токен"

In [4]:
# получаем данные по акциям
stocks = await get_stocks(TOKEN)

In [ ]:
# поскольку sdk позволяет выгружать данные только за 5 лет
# мы сформируем 2 датафрейма общей продолжительностью 9 лет
start_date = datetime.strptime('2016-01-01',"%Y-%m-%d")
end_date = datetime.strptime("2021-12-31", "%Y-%m-%d")
start_date_1 = datetime.strptime("2022-01-01", "%Y-%m-%d")
end_data_1= datetime.strptime("2025-12-31", "%Y-%m-%d")

# получаем свечи и объемы для тикеров в заданном диапазоне дат
df1 = await get_candles(TOKEN, stocks, start_date, end_date)
df2 = await get_candles(TOKEN, stocks, start_date_1, end_data_1)

# объединяем данные в один датафрейм, сортируем и сохраням его в CSV файл
df = pd.concat([df1, df2], ignore_index=True).sort_values(['ticker', 'date']).reset_index(drop=True)
df.to_csv('2016-2025_candles_to_train.csv')

Запрашиваем данные с 2016-01-01 по 2021-12-31
Всего тикеров: 152
Запрашиваем данные с 2022-01-01 по 2025-12-31
Всего тикеров: 152


In [13]:
# сформируем ещё один CSV фаил в котором будут свечи для окончательной проверки модели
start_date = datetime.strptime("2026-01-01", "%Y-%m-%d")
end_date = datetime.now() - timedelta(days=80)

df = await get_candles(TOKEN, stocks=stocks, tickers=stocks.keys(), start_date=start_date, end_date=end_date)
# сортируем
df = df.sort_values(['ticker', 'date']).reset_index(drop=True)
# сохраняем в фаил
df.to_csv('candles_to_validation.csv')

Запрашиваем данные с 2026-01-01 по 2026-08-11
Всего тикеров: 152


На выходе у нас появится 2 фаила на которых мы будем обучать и тестировать модель машинного обучения